# LLM Cyberbullying Detection: 0-shot vs Few-shot

This notebook implements task (a): compare **0-shot** and **few-shot** learning using classification **accuracy, precision, recall, and F1**.

In [ ]:
import os
import json
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from groq import Groq
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

LABELS = [
    'not_cyberbullying',
    'gender',
    'religion',
    'other_cyberbullying',
    'age',
    'ethnicity'
]

MODEL_NAME = 'llama-3.3-70b-versatile'
SAMPLE_SIZE = 100

db = pd.read_csv('./cyberbullying_tweets.csv')
db = db.dropna(subset=['tweet_text', 'cyberbullying_type']).copy()
db['cyberbullying_type'] = db['cyberbullying_type'].astype(str).str.strip().str.lower()
db = db[db['cyberbullying_type'].isin(LABELS)]

dotenv_path = Path('.env')
load_dotenv(dotenv_path=dotenv_path, override=True)
api_key = (os.getenv('GROQ_API_KEY') or '').strip().strip('"').strip("'")
if not api_key:
    raise ValueError('GROQ_API_KEY missing. Add it to HW3/.env')

client = Groq(api_key=api_key)

eval_df = db.iloc[:SAMPLE_SIZE].copy()
eval_df.head()

In [ ]:
SYSTEM_PROMPT = '''
You are an expert in cyberbullying detection.
Classify the tweet into exactly one label from:
not_cyberbullying, gender, religion, other_cyberbullying, age, ethnicity.
Return only valid JSON in the format: {"label": "one_label"}
'''.strip()

STATIC_EXAMPLES = [
    (
        "Need what he's smoking @RajAshok5 Being feminist isnt sexist BUT ASKING LAWS & INSISTING THAT WOMEN ARE CORRECT ALWAYS, MEN ARE CRIMINALS IS",
        "gender"
    ),
    (
        "@Raja5aab @Quickieleaks Yes, the test of god is that good or bad or indifferent or weird or whatever, it all proves gods existence.",
        "not_cyberbullying"
    ),
    (
        "In Islam women must be locked in their houses, and Muslims claim this is treating them well.",
        "religion"
    ),
    (
        "Girl who bullied me in high school just asked me for hair bleaching tips. Imma tell her 40 volume and to start at her roots",
        "age"
    )
]

def extract_label(response_text):
    if not isinstance(response_text, str):
        return 'error'
    try:
        payload = json.loads(response_text)
    except json.JSONDecodeError:
        return 'error'

    if not isinstance(payload, dict) or not payload:
        return 'error'

    for key in ('label', 'output', 'prediction', 'cyberbullying_type'):
        if key in payload and isinstance(payload[key], str):
            label = payload[key].strip().lower()
            return label if label in LABELS else 'error'

    first_value = str(next(iter(payload.values()))).strip().lower()
    return first_value if first_value in LABELS else 'error'

def call_model(messages, model_name=MODEL_NAME):
    completion = client.chat.completions.create(
        model=model_name,
        messages=messages,
        temperature=0,
        max_tokens=80,
        response_format={'type': 'json_object'}
    )
    return completion.choices[0].message.content

def build_zero_shot_messages(tweet):
    return [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': f'Tweet: {tweet}'}
    ]

def build_few_shot_messages(tweet):
    examples_text = []
    for ex_tweet, ex_label in STATIC_EXAMPLES:
        examples_text.append(f'Tweet: {ex_tweet}')
        examples_text.append(f'Answer: {{\"label\": \"{ex_label}\"}}')
    examples_block = '\n'.join(examples_text)

    return [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': f'{examples_block}\n\nNow classify this tweet:\nTweet: {tweet}'}
    ]

def run_method(tweets, message_builder, method_name):
    preds = []
    for i, tweet in enumerate(tweets, start=1):
        try:
            raw = call_model(message_builder(str(tweet)))
            preds.append(extract_label(raw))
        except Exception as e:
            print(f'{method_name} API error at row {i}: {e}')
            preds.append('error')
        if i % 20 == 0 or i == len(tweets):
            print(f'{method_name}: {i}/{len(tweets)} done')
    return preds

In [ ]:
tweets = eval_df['tweet_text'].astype(str).tolist()
y_true = eval_df['cyberbullying_type'].astype(str).str.strip().str.lower().tolist()

print('Running 0-shot...')
y_pred_zero_shot = run_method(tweets, build_zero_shot_messages, 'Zero-Shot')

print('Running few-shot...')
y_pred_few_shot = run_method(tweets, build_few_shot_messages, 'Few-Shot')

In [ ]:
def metric_row(name, y_true_vals, y_pred_vals):
    acc = accuracy_score(y_true_vals, y_pred_vals)
    p, r, f1, _ = precision_recall_fscore_support(
        y_true_vals, y_pred_vals, average='weighted', zero_division=0
    )
    return {
        'Method': name,
        'Accuracy': acc,
        'Precision (Weighted)': p,
        'Recall (Weighted)': r,
        'F1 (Weighted)': f1
    }

comparison_df = pd.DataFrame([
    metric_row('Zero-Shot', y_true, y_pred_zero_shot),
    metric_row('Few-Shot', y_true, y_pred_few_shot),
])

print('\n===== 0-shot vs Few-shot Performance =====')
comparison_df

## Interpretation Notes
- Higher **Accuracy** means more total correct classifications.
- Weighted **Precision/Recall/F1** account for class imbalance.
- If few-shot outperforms 0-shot, it indicates example-guided prompting helps this task.